In [13]:
import os
import wandb # для логирования

import numpy as np
import random
from tqdm import *
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim # для оптимизаторов
from torchvision import datasets # для данных
import torchvision.transforms as transforms # для преобразований тензоров
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn

import matplotlib.pyplot as plt

In [3]:
df= pd.read_csv('/Users/phuongnguyen/Downloads/car_policy.csv')

df.head()

,policy_tenure,age_of_car,age_of_policyholder,population_density,make,max_torque,max_power,airbags,is_esc,is_adjustable_steering,...,engine_type_K Series Dual jet,engine_type_K10C,engine_type_i-DTEC,rear_brakes_type_Drum,transmission_type_Manual,steering_type_Manual,steering_type_Power,safe_score,car_size,age_of_car_and_policy
0,0.515874,0.05,0.644231,4990,1,60.0,40.36,2,0,0,...,0,0,0,1,1,0,1,2,7698283125,0.025794
1,0.672619,0.02,0.375000,27003,1,60.0,40.36,2,0,0,...,0,0,0,1,1,0,1,2,7698283125,0.013452
2,0.841110,0.02,0.384615,4076,1,60.0,40.36,2,0,0,...,0,0,0,1,1,0,1,2,7698283125,0.016822
3,0.900277,0.11,0.432692,21622,1,113.0,88.50,2,1,1,...,0,0,0,1,0,0,0,6,10500957375,0.099030
4,0.596403,0.11,0.634615,34738,2,91.0,67.06,2,0,0,...,0,0,0,1,0,0,0,3,8777961010,0.065604


In [4]:
# Разделение на X и y
X = df.drop(columns = ['is_claim'])
y = df['is_claim']

print(X.shape)
print(y.shape)

(58592, 89)
(58592,)


In [5]:
y.value_counts()

is_claim
0    54844
1     3748
Name: count, dtype: int64

In [6]:
# Зафиксируем seed для воспроизводимости

def seed_everything(seed):
    random.seed(seed) # фиксируем генератор случайных чисел
    os.environ['PYTHONHASHSEED'] = str(seed) # фиксируем заполнения хешей
    np.random.seed(seed) # фиксируем генератор случайных чисел numpy
    torch.manual_seed(seed) # фиксируем генератор случайных чисел pytorch
    torch.cuda.manual_seed(seed) # фиксируем генератор случайных чисел для GPU
    #torch.backends.cudnn.deterministic = True # выбираем только детерминированные алгоритмы (для сверток)
    #torch.backends.cudnn.benchmark = False # фиксируем алгоритм вычисления сверток

In [ ]:
class CFG:

# Задаем параметры нашего эксперимента

  api = "----------"# вписать свой API Wandb
  project = "Models"# вписать название эксперимента, который предварительно надо создать в Wandb
  num_epochs = 10 # количество эпох
  train_batch_size = 64 # размер батча обучающей выборки
  test_batch_size = 512 # размер батча тестовой выборки
  num_workers = 2 # количество активных процессов на загрузку данных
  lr = 0.001 # learning_rate
  seed = 2022 # для функции воспроизводимости
  wandb = True # флаг использования Wandb

In [8]:
 #Поделим данные на train, test

X_train, X_test,  y_train, y_test  = train_test_split(X, y, test_size= 0.2, random_state = CFG.seed, stratify = y)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)


(46873, 89)
(11719, 89)
(46873,)
(11719,)


In [9]:
# Стандартизируем наги значения 
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#Превращаю в тензор
X_train_tens = torch.tensor(X_train_scaled, dtype  = torch.float32)
X_test_tens = torch.tensor(X_test_scaled, dtype  = torch.float32)
y_train_tens = torch.tensor(y_train.values, dtype  = torch.float32).reshape(-1, 1)
y_test_tens = torch.tensor(y_test.values, dtype  = torch.float32).reshape(-1, 1)


# Создаю train_dataset из X_train_tens и y_train_tens и  test_dataset из X_test_tens и y_test_tens
train_dataset = TensorDataset(X_train_tens, y_train_tens)
test_dataset = TensorDataset(X_test_tens, y_test_tens)

#создаю лоудары, чтобы передавались данные батчами
train_loader = DataLoader(train_dataset, batch_size= CFG.train_batch_size, shuffle= True, num_workers= CFG.num_workers)
test_loader = DataLoader(test_dataset, batch_size= CFG.test_batch_size, shuffle = False, num_workers= CFG.num_workers)




In [10]:
examples = enumerate(train_loader)

batch_ind, (example_data, example_targets ) = next(examples)

In [11]:
example_data.shape

torch.Size([64, 89])

In [12]:
example_targets.shape

torch.Size([64, 1])

## Модель 1 

In [ ]:


class Model_1 (nn.Module):
    
    def __init__(self):
        super(Model_1,self).__init__()
        
        hidden_1 = 256
        hidden_2 = 128
        hidden_3 = 64
        
        # первый слой (89 -> hidden_1)
        self.fc1 = nn.Linear(89, hidden_1)
        self.batch_norm1 = nn.BatchNorm1d(hidden_1)
        self.act1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.3)    
    
        # второй слой (hidden_1 -> hidden_2)
        self.fc2  = nn.Linear(hidden_1, hidden_2)
        self.batch_norm2 = nn.BatchNorm1d(hidden_2)
        self.act2 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.3)  
        
        # третий слой (hidden_2 -> hidden_3)
        self.fc3  = nn.Linear(hidden_2, hidden_3)
        self.batch_norm3 = nn.BatchNorm1d(hidden_3)
        self.act3 = nn.ReLU()
        self.dropout3 = nn.Dropout(0.2)
        
        #вывходной слой
        self.fc4 = nn.Linear(hidden_3, 1)
        
    def forward(self, x):
        
            
        x = self.fc1(x)
        x = self.batch_norm1(x)
        x = self.act1(x)
        x = self.dropout1(x)
            
            
        x = self.fc2(x)
        x = self.batch_norm2(x)
        x = self.act2(x)
        x = self.dropout2(x)
            
        x = self.fc3(x)
        x = self.batch_norm3(x)
        x = self.act3(x)
        x = self.dropout3(x)
            
        x = self.fc4(x)
            
        return x

In [33]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = Model_1().to(device)

print(device)
print(model)

cpu
Model_1(
  (fc1): Linear(in_features=89, out_features=256, bias=True)
  (batch_norm1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (act1): ReLU()
  (dropout1): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=256, out_features=128, bias=True)
  (batch_norm2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (act2): ReLU()
  (dropout2): Dropout(p=0.3, inplace=False)
  (fc3): Linear(in_features=128, out_features=64, bias=True)
  (batch_norm3): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (act3): ReLU()
  (dropout3): Dropout(p=0.2, inplace=False)
  (fc4): Linear(in_features=64, out_features=1, bias=True)
)


In [35]:
# функция потерь 
criterion = nn.BCEWithLogitsLoss()

#оптимизатор
optimizer = torch.optim.Adam(model.parameters(), lr = CFG.lr) #https://docs.pytorch.org/docs/main/generated/torch.optim.Adam.html

In [ ]:

#крч проверяю на одном батче как проходит и считает loss
example_data = example_data.to(device)
example_targets = example_targets.to(device)

outputs = model(example_data)

loss = criterion(outputs, example_targets)

print(outputs.shape)
print(example_targets.shape)
print(loss.item())

torch.Size([64, 1])
torch.Size([64, 1])
0.6901258826255798


In [ ]:
# функция обучения модели

